In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
import pickle
%matplotlib inline

In [5]:
#Loading dataset
df = pd.read_csv(r"C:\Users\david.sheridan\Downloads\insurance.csv")

In [7]:
#Basic cleanup
df = df.dropna()

# Defining target and predictors
y = df['insurance_cost']
X = df.drop('insurance_cost', axis=1)

#Checking column types and choose transformations
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

#Preprocessing pipeline
preprocess_pipeline = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop='first', sparse_output=False), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ],
    remainder='passthrough'
).set_output(transform="pandas")

In [9]:
#Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
#Defining Dimensionality Reduction Options
N_FEATURES_OPTIONS = [2, 5, 10]

dim_red_methods = [
    ('pca', PCA()),
    ('rfe_lr', RFE(LinearRegression())),
    ('select_k_best', SelectKBest(score_func=f_regression))
]

In [15]:
#Defining and Training 3 Pipelines via GridSearchCV
#A. Random Forest Pipeline
pipe_rf = Pipeline([
    ('preprocess', preprocess_pipeline),
    ('reduce_dim', 'passthrough'),
    ('ttr', TransformedTargetRegressor(
        regressor=RandomForestRegressor(n_estimators=10),
        func=np.log1p,
        inverse_func=np.expm1))
])

param_grid_rf = [
    {
        'reduce_dim': [PCA()],
        'reduce_dim__n_components': N_FEATURES_OPTIONS,
        'ttr__regressor__max_depth': [3, 5, 7]
    },
    {
        'reduce_dim': [RFE(estimator=LinearRegression())],
        'reduce_dim__n_features_to_select': N_FEATURES_OPTIONS,
        'ttr__regressor__max_depth': [3, 5, 7]
    },
    {
        'reduce_dim': [SelectKBest(score_func=f_regression)],
        'reduce_dim__k': N_FEATURES_OPTIONS,
        'ttr__regressor__max_depth': [3, 5, 7]
    }
]

search_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, n_jobs=-1)
search_rf.fit(X_train, y_train)

C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
15 fits failed out of a total of 135.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\david.sheridan\AppData\Local\ana

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['gender',
                                                                          'smoker',
                                                                          'region']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['age',
                                                                          'bmi',
                                                                          'children'])])),
                                       ('reduce_dim', 'passthrough'),
                                       ('ttr',
                                        TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                                                   inverse_func=<u...
                          'reduce_dim__n_components': [2, 5, 10],
                          'ttr__regressor__max_depth': [3, 5, 7]},
                         {'reduce_dim': [RFE(estimator=LinearRegression())],
                          'reduce_dim__n_features_to_select': [2, 5, 10],
                          'ttr__regressor__max_depth': [3, 5, 7]},
                         {'reduce_dim': [SelectKBest(score_func=<function f_regression at 0x0000023C0080ED40>)],
                          'reduce_dim__k': [2, 5, 10],
                          'ttr__regressor__max_depth': [3, 5, 7]}])

In [17]:
#B. Linear Regression Pipeline
pipe_lr = Pipeline([
    ('preprocess', preprocess_pipeline),
    ('reduce_dim', 'passthrough'),
    ('ttr', TransformedTargetRegressor(
        regressor=LinearRegression(),
        func=np.log1p,
        inverse_func=np.expm1))
])

param_grid_lr = [
    {
        'reduce_dim': [PCA()],
        'reduce_dim__n_components': N_FEATURES_OPTIONS
    },
    {
        'reduce_dim': [RFE(estimator=LinearRegression())],
        'reduce_dim__n_features_to_select': N_FEATURES_OPTIONS
    },
    {
        'reduce_dim': [SelectKBest(score_func=f_regression)],
        'reduce_dim__k': N_FEATURES_OPTIONS
    }
]

search_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, n_jobs=-1)
search_lr.fit(X_train, y_train)

C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
5 fits failed out of a total of 45.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\david.sheridan\AppData\Local\anacon

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['gender',
                                                                          'smoker',
                                                                          'region']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['age',
                                                                          'bmi',
                                                                          'children'])])),
                                       ('reduce_dim', 'passthrough'),
                                       ('ttr',
                                        TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                                                   inverse_func=<ufunc 'expm1'>,
                                                                   regressor=LinearRegression()))]),
             n_jobs=-1,
             param_grid=[{'reduce_dim': [PCA()],
                          'reduce_dim__n_components': [2, 5, 10]},
                         {'reduce_dim': [RFE(estimator=LinearRegression())],
                          'reduce_dim__n_features_to_select': [2, 5, 10]},
                         {'reduce_dim': [SelectKBest(score_func=<function f_regression at 0x0000023C0080ED40>)],
                          'reduce_dim__k': [2, 5, 10]}])

In [19]:
#C.SVR Pipeline
pipe_svr = Pipeline([
    ('preprocess', preprocess_pipeline),
    ('reduce_dim', 'passthrough'),
    ('ttr', TransformedTargetRegressor(
        regressor=SVR(kernel='rbf'),
        func=np.log1p,
        inverse_func=np.expm1))
])

param_grid_svr = [
    {
        'reduce_dim': [PCA()],
        'reduce_dim__n_components': N_FEATURES_OPTIONS,
        'ttr__regressor__C': [0.1, 1, 10]
    },
    {
        'reduce_dim': [RFE(estimator=LinearRegression())],
        'reduce_dim__n_features_to_select': N_FEATURES_OPTIONS,
        'ttr__regressor__C': [0.1, 1, 10]
    },
    {
        'reduce_dim': [SelectKBest(score_func=f_regression)],
        'reduce_dim__k': N_FEATURES_OPTIONS,
        'ttr__regressor__C': [0.1, 1, 10]
    }
]

search_svr = GridSearchCV(pipe_svr, param_grid_svr, cv=5, n_jobs=-1)
search_svr.fit(X_train, y_train)

C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
15 fits failed out of a total of 135.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\david.sheridan\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\david.sheridan\AppData\Local\ana

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['gender',
                                                                          'smoker',
                                                                          'region']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['age',
                                                                          'bmi',
                                                                          'children'])])),
                                       ('reduce_dim', 'passthrough'),
                                       ('ttr',
                                        TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                                                   inverse_func=<u...
             param_grid=[{'reduce_dim': [PCA()],
                          'reduce_dim__n_components': [2, 5, 10],
                          'ttr__regressor__C': [0.1, 1, 10]},
                         {'reduce_dim': [RFE(estimator=LinearRegression())],
                          'reduce_dim__n_features_to_select': [2, 5, 10],
                          'ttr__regressor__C': [0.1, 1, 10]},
                         {'reduce_dim': [SelectKBest(score_func=<function f_regression at 0x0000023C0080ED40>)],
                          'reduce_dim__k': [2, 5, 10],
                          'ttr__regressor__C': [0.1, 1, 10]}])

In [21]:
#Evaluating All Models
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'MSE': mean_squared_error(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }

results = {
    'Random Forest': evaluate_model(search_rf.best_estimator_, X_test, y_test),
    'Linear Regression': evaluate_model(search_lr.best_estimator_, X_test, y_test),
    'SVR': evaluate_model(search_svr.best_estimator_, X_test, y_test)
}

pd.DataFrame(results)

,Random Forest,Linear Regression,SVR
MSE,1.708909e+07,7.141904e+07,2.051529e+07
MAE,2.020888e+03,4.349935e+03,2.153672e+03
R2,8.790143e-01,4.943744e-01,8.547578e-01


In [25]:
#Saving The Best Model 
best_model = search_rf if results['Random Forest']['R2'] == max(r['R2'] for r in results.values()) else \
             search_lr if results['Linear Regression']['R2'] == max(r['R2'] for r in results.values()) else \
             search_svr

pickle.dump(best_model.best_estimator_, open('final_model_regression.sav', 'wb'))

In this regression task, I trained and evaluated three models—Random Forest, Linear Regression, and SVR—to predict insurance costs. Each model was embedded in a pipeline that included preprocessing and dimensionality reduction (PCA, RFE, and SelectKBest). Hyperparameters were optimized using GridSearchCV. Among the models, Random Forest achieved the best performance in terms of R², MAE, and MSE, making it the most accurate predictor. This exercise deepened my understanding of integrating dimensionality reduction into regression pipelines and reinforced the importance of model evaluation through systematic cross-validation.
